#  APLICACIÓN DEL MODELO Y ESTIMACIÓN DEL AHORRO A NIVEL  PAÍS


Esta notebook desarrolla el componente de Ciencia de Datos de una solución destinada a analizar el consumo energético residencial, evaluar la eficiencia de los hogares, identificar oportunidades de ahorro y estimar su impacto económico a escala poblacional.

Utiliza el modelo ajustado anterirormente para clasificar a los hogares según su eficiencia y consumo. Luego selecciona solo los hogares que tengan menor eficiencia (40% de los hogares).

En tanto al consumo, se divide a los hogares en el 40% de mayor consumo promedio, 20% central y 40% de menor consumo promedio. Entendiendo al "consumo promedio" como el consumo esperado para cada tipo de hogar.

Luego, basandose en el consumo que demuestren en cada categoría de consumo (aire acondicionado, calefacción, cocinar, iluminacción, etc) estima el ahorro potencial en kWh.

Finalmente utiliza factores de expansión para poder evaluar el efecto real de estas medidas.

# RESULTADOS

- De aplicar todas las recomendaciones, para Estados Unidos se encontraron oportunidades de ahorro de entre US\$ 360 y US\$ 920  por hogar.

- Para el caso de Chile la oportunidad e ahorro variaba entre US\$ 150 y US\$ 300.

- La diferencia entre ambos paises probablemente se deba al extenso uso de la calefacción a gas en Chile junto al desuso de aire acondicionado, resultando naturalmente en un menor consumo electricto y menor impacto de las medidas de ahorro.

- A nivel pais, estas medidas podrían representar un ahorro de US\$ 30,077 millones y US\$ 561 millones en Estados Unidos y Chile respectivamente.

# IMPORTS

In [1]:
import numpy as np
import pandas as pd

# ESTADOS UNIDOS

## CARGA LA BASE

In [22]:
# Carga dataset
##Rama Main:
#url = "https://raw.githubusercontent.com/No-Country-simulation/Team45-G9/refs/heads/main/bases/USA/recs2015_public_v4.csv"
## Rama data_science:
url = "https://raw.githubusercontent.com/No-Country-simulation/Team45-G9/refs/heads/data_science/data_science/bases/EEUU/recs2015_public_v4.csv"

data = pd.read_csv(url)

print(data.head())
print(data.info())

   DOEID  REGIONC  DIVISION METROMICRO UATYP10  TYPEHUQ  ZTYPEHUQ  CELLAR  \
0  10001        4        10      METRO       U        2         0       0   
1  10002        3         7       NONE       R        2         0       0   
2  10003        3         6      METRO       U        2         0       1   
3  10004        2         4      MICRO       C        2         0       1   
4  10005        1         2      METRO       U        2         0       1   

   ZCELLAR  BASEFIN  ...  ZELAMOUNT  NGXBTU  PERIODNG  ZNGAMOUNT  FOXBTU  \
0        0       -2  ...          0  103.32         1          0  137.45   
1        0       -2  ...          1     NaN        -2         -2  137.45   
2        0        1  ...          0  100.14         1          0  137.45   
3        0        1  ...          0     NaN        -2         -2  137.45   
4        0        0  ...          0  102.83         1          0  137.45   

   PERIODFO  ZFOAMOUNT  LPXBTU  PERIODLP  ZLPAMOUNT  
0        -2         -2   9

## LIMPIA LA BASE

In [23]:
# Crea un dataset vacío
data_1 = pd.DataFrame(index=data.index)

#Consumo
data_1['KWH'] = data['KWH']
#Pesos
data_1['NWEIGHT'] = data['NWEIGHT']

data_1['CLIMA_grados_dias_calefaccion_log'] = np.log10(1+ data["HDD65"])
data_1['CLIMA_grados_dias_enfriamiento_log'] = np.log10(1+  data["CDD65"] )
#

map={10: 1.5, 20:4.0 , 30:7.5 , 41:12.5 , 42:17.5, 50:24.5, 60:34.5}
data_1['VENTANAS_log'] = np.log10(1+  data['WINDOWS'].map(map) )


#Electricidad

# Electrodomésticos
data_1['REFRIGERADOR_log'] = np.log10(1+ data['NUMFRIG'])
data_1['FREEZER_log'] = np.log10(1+ data['NUMFREEZ'])
data_1['HORNO_electrico'] = (data['STOVENFUEL']==5).astype(int)

data_1['LAVARROPAS_frecuencia_log'] =np.log10( 1 +  data['WASHLOAD'].clip(lower=0) )
data_1['SECARROPAS_electrico'] = (data['DRYRFUEL']==5).astype(int)
data_1['TV_log'] = np.log10(1+  data['TVCOLOR'])

map = {1:2.5 , 2:10.0 , 3:25.0 , 4:42.5 ,5:55.0 ,  -2:0.0}
TV_frecuencia_semana_laboral = data['TVONWD1'].map(map)
map = {1:1.0 , 2:4.0 , 3:10.0 , 4:17.0 ,5:22.0 ,  -2:0.0}
TV_frecuencia_fin_de_semana = data['TVONWE1'].map(map)
data_1['TV_frecuencia_semana_log'] = np.log10(1+ TV_frecuencia_semana_laboral + TV_frecuencia_fin_de_semana )

# Agua
data_1['AGUA_CALIENTE_electrica'] = (data['FUELH2O']==5).astype(int)
# LUCES
data_1['LUCES_4_horas_log'] = np.log10(1+  data['LGTIN4'] )
map={ 0: 0.0, 1:2.5, 2: 7.0, 3: 12.5, -2:0.0}
data_1['LUCES_afuera_log'] = np.log10(1+ data['LGTOUTNUM'].map(map) )

## Habitaciones
data_1['DORMITORIOS_log'] = np.log10(1+ data['BEDROOMS'])

# AIRE ACONDICIONADO
data_1['AIRE_ACONDICIONADO'] = (data['AIRCOND']==1).astype(int)

# CALEFACCION
data_1['CALEFACCION_electrico'] = (data['FUELHEAT'] == 5).astype(int)
# Integrantes del hogar

data_1['HABITANTES_mayores_log'] = np.log10(1+ data['NUMADULT'])
data_1['HABITANTES_menores_log'] = np.log10(1+ data['NUMCHILD'])

# CATEGORIAS DE CONSUMO

data_1['KWH_AGUA_SANITARIA'] = data['KWHWTH']

data_1['KWH_AIRE_ACONDICIONADO'] = data['KWHCOL']
data_1['KWH_CALEFACCION'] = data['KWHSPH']
data_1['KWH_COCINA'] = (data['KWHMICRO'] + data['KWHCOK'])
data_1['KWH_REFRIGERACION'] = (data['KWHRFG'] + data['KWHFRZ'])
data_1['KWH_LAVADO'] = (data['KWHCW'] + data['KWHCDR'])

data_1['KWH_ILUMINACION'] = data['KWHLGT']

data_1['KWH_TV'] = data['KWHTVREL']
data_1['KWH_OTROS'] = (data['KWHNEC'] + data['KWHHTBHEAT']
                             + data['KWHHTBPMP'] + data['KWHPLPMP']
                             + data['KWHHUM'] + data['KWHDHUM']
                              + data['KWHCFAN'] + data['KWHEVAPCOL']
                              + data['KWHAHUCOL'] + data['KWHAHUHEAT']
                             + data['KWHDWH']
            )




## CARGA EL MODELO

In [24]:
MODELO_USA = {'const': 2.5929737400729156,
           'CLIMA_grados_dias_enfriamiento_log': 0.09954211071782615,
           'CLIMA_grados_dias_calefaccion_log': 0.0068828117583398934,
           'DORMITORIOS_log': 0.49423717926161864,
           'VENTANAS_log': 0.2980296648011681,
           'HABITANTES_mayores_log': 0.3753245275669875,
           'HABITANTES_menores_log': 0.11633833178520547,
           'AGUA_CALIENTE_electrica': 0.13578276769858272,
           'CALEFACCION_electrico': 0.141543247825371,
           'AIRE_ACONDICIONADO': 0.1496148110403177}

## APLICA EL MODELO

In [25]:
data_1['const'] = 0
prediccion = data_1['const']
data_1['const'] = 1
for var in MODELO_USA.keys():
  prediccion += MODELO_USA[var] * data_1[var]

data_1['PREDICCION'] = 10**prediccion

In [26]:
data_1['Eficiencia'] = 'Muy alta'
data_1['Consumo'] = 'Muy bajo'

data_1.loc[data_1["PREDICCION"] >  4883, 'Consumo']= 'Bajo'
data_1.loc[data_1["PREDICCION"] >  7722, 'Consumo']= 'Moderado'
data_1.loc[data_1["PREDICCION"] >  10945, 'Consumo']= 'Alto'
data_1.loc[data_1["PREDICCION"] >  15788, 'Consumo']= 'Muy alto'


z = ( np.log10(1+data_1["KWH"]) - np.log10(1+data_1["PREDICCION"]) ) / 0.2118
data_1.loc[z > -0.84, 'Eficiencia']= 'Alta'
data_1.loc[z > -0.25, 'Eficiencia']= 'Moderada'
data_1.loc[z > 0.25, 'Eficiencia']= 'Baja'
data_1.loc[z > 0.84, 'Eficiencia']= 'Muy baja'




data_1['AHORRO']  = 0


# ============================================================
# RECOMENDACIONES ESPECÍFICAS
# ============================================================
# ------------------------------------------------------------
# AGUA CALIENTE
# ------------------------------------------------------------
aux =  data_1['AGUA_CALIENTE_electrica'] == 1
data_1.loc[aux, 'AHORRO'] += data_1.loc[aux, 'KWH_AGUA_SANITARIA'] * 0.20

# ------------------------------------------------------------
# AIRE ACONDICIONADO
# ------------------------------------------------------------
aux =  data_1['KWH_AIRE_ACONDICIONADO'] > 0.1 * data_1['KWH']
data_1.loc[aux, 'AHORRO'] += data_1.loc[aux, 'KWH_AIRE_ACONDICIONADO'] * 0.15

# ------------------------------------------------------------
# CALEFACCIÓN
# ------------------------------------------------------------
aux =  data_1['KWH_CALEFACCION'] > 0.1 * data_1['KWH']
data_1.loc[aux, 'AHORRO'] += data_1.loc[aux, 'KWH_CALEFACCION'] * 0.20

# ------------------------------------------------------------
# VENTANAS
# ------------------------------------------------------------
aux = (10**data_1['VENTANAS_log'] - 1) > (3 * (10**data_1['DORMITORIOS_log'] - 1))
data_1.loc[aux, 'AHORRO'] += data_1.loc[aux, 'KWH_CALEFACCION'] * 0.25 + data_1.loc[aux, 'KWH_AIRE_ACONDICIONADO'] * 0.15

# ------------------------------------------------------------
# SECARROPAS / LAVADO
# ------------------------------------------------------------

aux = (data_1['SECARROPAS_electrico'] == 1) & (data_1['LAVARROPAS_frecuencia_log'] >= np.log10(2.5+1))
data_1.loc[aux, 'AHORRO'] += data_1.loc[aux, 'KWH_LAVADO'] * 0.5

# ------------------------------------------------------------
# STANDBY — SOLO CUANDO NO HAY GRANDES CONSUMIDORES
# ------------------------------------------------------------
aux = (data_1['AIRE_ACONDICIONADO']==0) & (data_1['CALEFACCION_electrico']==0) & (data_1['AGUA_CALIENTE_electrica']==0)
data_1.loc[aux, 'AHORRO'] += data_1.loc[aux, 'KWH']*0.10

#--------------------------------------------------
# ILUMINACIÓN
# ------------------------------------------------------------
aux = data_1['KWH_ILUMINACION'] > 0.1 * data_1['KWH']
data_1.loc[aux, 'AHORRO'] += data_1.loc[aux, 'KWH_ILUMINACION'] * 0.5

#-------------------------------------------------
# REFRIGERACION
#-------------------------------------------------

aux = data_1['KWH_REFRIGERACION'] > 0.1 * data_1['KWH']
data_1.loc[aux, 'AHORRO'] += data_1.loc[aux, 'KWH_REFRIGERACION'] * 0.3

ref = 10**data_1['REFRIGERADOR_log'] - 1
frz = 10**data_1['FREEZER_log'] - 1
aux = (data_1['KWH_REFRIGERACION'] > 0.1 * data_1['KWH']) & ((ref>1.5) |  (frz>0.5) )
data_1.loc[aux, 'AHORRO'] += ( ref.loc[aux] + frz.loc[aux] - 1 ) * 400












/tmp/ipykernel_1983/2202898807.py:29: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[417.9358 667.0602 632.9282 ... 191.9486 371.7438 258.379 ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data_1.loc[aux, 'AHORRO'] += data_1.loc[aux, 'KWH_AGUA_SANITARIA'] * 0.20


## AHORRO POR GRUPO







In [27]:
def get_ahorro(data, consumo, eficiencia, fun='sum'):
  if isinstance(consumo, str):
    consumo = [consumo]
  if isinstance(eficiencia, str):
    eficiencia = [eficiencia]
  temp_data = data.loc[(data['Consumo'].isin(consumo) ) & (data['Eficiencia'].isin(eficiencia)), ['AHORRO', 'NWEIGHT']]
  out = temp_data['AHORRO'] * temp_data['NWEIGHT']
  if fun == 'sum':
    out = out.sum()
  elif fun == 'mean':
    out = out.sum() / temp_data['NWEIGHT'].sum()
  else:
    print('Función no reconocida')

  return out

print("AHORRO POR GRUPO")
print("--------------")
print("CONSUMO BAJO")
print( f"Ineficientes: US${ get_ahorro(data_1, ["Muy bajo", "Bajo"], ["Muy baja", "Baja"], 'mean')*0.188:,.0f}")
print("CONSUMO MEDIO")
print( f"Ineficientes: US${ get_ahorro(data_1, ["Moderado"], ["Muy baja", "Baja"], 'mean')*0.188:,.0f}")
print("CONSUMO ALTO")
print( f"Ineficientes: US${ get_ahorro(data_1, ["Muy alto", "Alto"], ["Muy baja", "Baja"], 'mean')*0.188:,.0f}")


AHORRO POR GRUPO
--------------
CONSUMO BAJO
Ineficientes: US$362
CONSUMO MEDIO
Ineficientes: US$583
CONSUMO ALTO
Ineficientes: US$921


##AHORRO PAIS

In [28]:

print("AHORRO PAIS")
print("--------------")
print("TOTAL")
print( f"Ineficientes: US${ get_ahorro(data_1, ["Muy bajo", "Bajo", "Moderado","Muy alto", "Alto"], ["Muy baja", "Baja"], 'sum')*0.188:,.0f}")
print("--------------")
print("CONSUMO BAJO")
print( f"Ineficientes: US${ get_ahorro(data_1, ["Muy bajo", "Bajo"], ["Muy baja", "Baja"], 'sum')*0.188:,.0f}")
print("-------------")
print("CONSUMO MEDIO")
print( f"Ineficientes: US${ get_ahorro(data_1, ["Moderado"], ["Muy baja", "Baja"], 'sum')*0.188:,.0f}")
print("-------------")
print("CONSUMO ALTO")
print( f"Ineficientes: US${ get_ahorro(data_1, ["Alto", "Muy alto"], ["Muy baja", "Baja"], 'sum')*0.188:,.0f}")


AHORRO PAIS
--------------
TOTAL
Ineficientes: US$30,077,657,727
--------------
CONSUMO BAJO
Ineficientes: US$6,489,046,361
-------------
CONSUMO MEDIO
Ineficientes: US$9,049,053,769
-------------
CONSUMO ALTO
Ineficientes: US$14,539,557,598


# CHILE

## CARGA LA BASE

In [9]:
# Carga dataset
url = "https://raw.githubusercontent.com/No-Country-simulation/Team45-G9/data_science/data_science/bases/CHILE/bbdd_estudio_caracterizacion_residencial_2018_1.xlsx"
data = pd.read_excel(url)


# Preview
print(data.head())

   Folio  Clasificación_socioeconómica   F1   F2  P1  P2  P3_A  P3_B  P4  P5  \
0      1                             1  1.0  2.0   1 NaN   NaN   NaN   2   2   
1      2                             1  2.0  2.0   1 NaN   NaN   NaN   2   3   
2      3                             2  1.0  2.0   1 NaN   NaN   NaN   2   4   
3      4                             3  1.0  2.0   1 NaN   NaN   NaN   2   2   
4      5                             1  1.0  2.0   1 NaN   NaN   NaN   2   2   

   ...  Aspiradora Electricidad  Computador Electricidad  TV Electricidad  \
0  ...             63776.272095             26262.741791    203063.650351   
1  ...                 0.000000             34465.242092    275189.721288   
2  ...             54846.558931             84695.743888    134703.148734   
3  ...                 0.000000              3936.971598    348538.183457   
4  ...                 0.000000             16282.767975    385787.695972   

   Juegos Electricidad Stand by Electricidad  A/C Electr

## LIMPIA LA BASE

In [10]:
# Crea un dataset vacío
data_1 = pd.DataFrame(index=data.index)

#Consumo

# consumo electrico
vars_consumo= ['Ducha Elec' ,'Tina Elec', 'Lavado loza Elec', 'Cocina + Horno Elec' ,
               'Lavado Ropa Elec', 'Secado Ropa Elec',
       'Calefacción Central Elec','Calefactores Elec',  'Microondas Electricidad',
       'Hornillo elec. Electricidad','Iluminación Electricidad',
       'Refrigerador Electricidad','Freezer Electricidad','Hervidor Electricidad',
       'Plancha Electricidad','Aspiradora Electricidad','Computador Electricidad',
       'TV Electricidad','Juegos Electricidad','Stand by Electricidad','A/C Electricidad',
       'Cafetera Electricidad','Bomba de riego Electricidad','Piscina Electricidad','Otros Electricidad',]
data_1['KWH'] = data[vars_consumo].sum(axis=1) / data['FE']

#Pesos
data_1['NWEIGHT'] = data['FE']


#

map = {1:1331 , 2:1348, 3:1111, 4:1834, 5: 2208, 6: 2403, 7:4039}
data_1['CLIMA_grados_dias_calefaccion_log'] =  np.log10( 1 +data['ZT'].map(map)  )

map = {1:33 , 2:90, 3:413, 4:23, 5: 57, 6: 10, 7:0}
data_1['CLIMA_grados_dias_enfriamiento_log'] =  np.log10( 1 +data['ZT'].map(map)  )
#

vars_ventanas = ['P10_A_Pequeño', 'P10_A_Mediano' ,'P10_A_Grande',
                 'P10_B_Pequeño', 'P10_B_Mediano', 'P10_B_Grande' ]
data_1['VENTANAS_log'] = np.log10( 1 + data[vars_ventanas].sum(axis=1) )





#Electricidad








# Electrodomésticos
data_1['REFRIGERADOR_log'] = np.log10(1+data['P94'].fillna(0))
data_1['FREEZER_log'] = np.log10( 1 + data['P98'].fillna(0) )
data_1['HORNO_electrico'] = (data['P54'] == 4).astype(int)
data_1['SECARROPAS_electrico'] = (data['P116'] == 2 ).astype(int)

# Agua
data_1['AGUA_CALIENTE_electrica'] = (data['P40_A'] == 4 ).astype(int)
# LUCES
# Se suman todas las luces para habitaciones que tengan al menos 4 horas de iluminación
# Se suman todas las horas de las luces y se las divide por 4.
vars_luces_A = ["P93_Living_A" , "P93_Comedor_A" , "P93_Cocina_A" ,   "P93_Baños_A"  ,  "P93_Dormitorios_A"  , "P93_Pasillos_A" ]
vars_luces_C = ["P93_Living_C" , "P93_Comedor_C" , "P93_Cocina_C" ,   "P93_Baños_C"  ,  "P93_Dormitorios_C"  , "P93_Pasillos_C" ]

data_1['LUCES_4_horas_log'] = 0
for i in range(len(vars_luces_A)):
    cantidad_de_luces_mas_de_4 = data[vars_luces_A[i]].fillna(0) * (data[vars_luces_C[i]].fillna(0)>4)
    horas_de_luces = data[vars_luces_C[i]].fillna(0)
    data_1['LUCES_4_horas_log'] += np.minimum(cantidad_de_luces_mas_de_4, horas_de_luces / 4)
data_1['LUCES_4_horas_log'] = np.log10( 1 + data_1['LUCES_4_horas_log'] )

data_1['LUCES_afuera_log'] = np.log10( 1 + data['P93_Patios_A'].fillna(0))



# AIRE ACONDICIONADO
data_1['AIRE_ACONDICIONADO'] = (data['P87']==1).astype(int)
# CALEFACCION
data_1['CALEFACCION_electrico'] = (data['P61'] == 5 ).astype(int)

# Integrantes del hogar
data_1['HABITANTES_totales_log'] = np.log10(1 + data['P153'] )


# CATEGORIAS DE CONSUMO
'''
data_1['KWH_AGUA_SANITARIA'] = data['KWHWTH']

data_1['KWH_AIRE_ACONDICIONADO'] = data['KWHCOL']
data_1['KWH_CALEFACCION'] = data['KWHSPH']
data_1['KWH_COCINA'] = (data['KWHMICRO'] + data['KWHCOK'])
data_1['KWH_REFRIGERACION'] = (data['KWHRFG'] + data['KWHFRZ'])
data_1['KWH_LAVADO'] = (data['KWHCW'] + data['KWHCDR'])
data_1['KWH_ILUMINACION'] = data['KWHLGT']

data_1['KWH_TV'] = data['KWHTVREL']
data_1['KWH_OTROS'] = (data['KWHNEC'] + data['KWHHTBHEAT']
                             + data['KWHHTBPMP'] + data['KWHPLPMP']
                             + data['KWHHUM'] + data['KWHDHUM']
                              + data['KWHCFAN'] + data['KWHEVAPCOL']
                              + data['KWHAHUCOL'] + data['KWHAHUHEAT']
                             + data['KWHDWH']
            )
'''
####################################################
###################################################
###################################################


# DEMASIADOS MISSING
#model_data['HABITANTES_menores_log'] = np.log10(1 +data['151.1'].fillna(0) + data['151.2'].fillna(0) )
#model_data['HABITANTES_mayores_log'] = np.log10(1 + data['151.3'].fillna(0) + data['151.4'].fillna(0) + data['151.5'].fillna(0) )

data_1['DORMITORIOS_log'] = 0 ## PENDIENTE DE IMPUTACION? USAR METROS CUADRADOS COMO REFERENCIA?




# HDD y CDD



# Luces
#Refrigracion


#Lavarropas (promediar verano e invierno; ignorar na; reemplazar 99 por las medianas)

### Estima el número de habitaciones basado en el tamaño del hogar
#Modelo basado en estados unidos ; 1sqft = 092903 m2

data_1['DORMITORIOS_log'] = 4


data_1.loc[data['P7'] <= 2440.0*0.092903, 'DORMITORIOS_log'] = 3
data_1.loc[data['P7'] <= 1159.5*0.092903, 'DORMITORIOS_log'] = 2
data_1.loc[data['P7'] <= 777.5*0.092903, 'DORMITORIOS_log'] = 1
data_1.loc[data['P93_Dormitorios_A'].isna() , 'DORMITORIOS_log'] = 0

data_1['DORMITORIOS_log'] = np.log10( 1 + data_1['DORMITORIOS_log'] )


#LAVARROPAS

#Lavarropas (promediar verano e invierno; ignorar na; reemplazar 99 por las medianas)
p110 = data['P110']
p111 = data['P111']

# Valid values are neither NA nor 99
valid_values = pd.concat([
    p110[p110.notna() & (p110 != 99)],
    p111[p111.notna() & (p111 != 99)]
])

median_lavado = valid_values.median()

# Conditions
p110_valid = p110.notna() & (p110 != 99)
p111_valid = p111.notna() & (p111 != 99)

p110_na = p110.isna()
p111_na = p111.isna()

p110_99 = p110.eq(99)
p111_99 = p111.eq(99)

# Start with 0: both are NA
data_1['LAVARROPAS_frecuencia_log'] = 0.0

# Both valid -> average
both_valid = p110_valid & p111_valid
data_1.loc[both_valid, 'LAVARROPAS_frecuencia_log'] = (
    (p110[both_valid] + p111[both_valid]) / 2
)

# Only P110 valid
only_p110 = p110_valid & ~p111_valid & ~p111_99
data_1.loc[only_p110, 'LAVARROPAS_frecuencia_log'] = p110[only_p110]

# Only P111 valid
only_p111 = p111_valid & ~p110_valid & ~p110_99
data_1.loc[only_p111, 'LAVARROPAS_frecuencia_log'] = p111[only_p111]

# One is NA and the other is 99 -> median
median_cases = (
    (p110_na & p111_99) |
    (p111_na & p110_99) |
    (p110_99 & p111_99)
)

data_1.loc[median_cases, 'LAVARROPAS_frecuencia_log'] = median_lavado


data_1['LAVARROPAS_frecuencia_log'] = np.log10( 1 + data_1['LAVARROPAS_frecuencia_log']  )







#INtERACCIONES

data_1['AIRE_ACONDICIONADO_X_CLIMA_grados_dias_enfriamiento_log'] =  data_1['AIRE_ACONDICIONADO'] * data_1['CLIMA_grados_dias_enfriamiento_log']
data_1['VENTANAS_log_X_CLIMA_grados_dias_calefaccion_log'] =  data_1['VENTANAS_log'] * data_1['CLIMA_grados_dias_calefaccion_log']
data_1['CALEFACCION_electrico_X_CLIMA_grados_dias_calefaccion_log'] =  data_1['CALEFACCION_electrico'] * data_1['CLIMA_grados_dias_calefaccion_log']


data_1['LUCES_afuera_log_X_LUCES_4_horas_log'] =  data_1['LUCES_afuera_log'] * data_1['LUCES_4_horas_log']

data_1['VENTANAS_log_X_CLIMA_grados_dias_calefaccion_log']= data_1['VENTANAS_log']*data_1['CLIMA_grados_dias_calefaccion_log']
data_1['VENTANAS_log_X_CLIMA_grados_dias_enfriamiento_log']= data_1['VENTANAS_log']*data_1['CLIMA_grados_dias_enfriamiento_log']
data_1['VENTANAS_log_X_DORMITORIOS_log']= data_1['VENTANAS_log']*data_1['DORMITORIOS_log']
data_1['DORMITORIOS_log_X_HABITANTES_totales_log']= data_1['HABITANTES_totales_log']*data_1['DORMITORIOS_log']




# efecto cuadrático
data_1['CLIMA_grados_dias_calefaccion_log_2'] = data_1["CLIMA_grados_dias_calefaccion_log"]**2
data_1['CLIMA_grados_dias_enfriamiento_log_2'] = data_1['CLIMA_grados_dias_enfriamiento_log']**2
data_1['LUCES_4_horas_log_2'] = data_1['LUCES_4_horas_log']**2

################################################################
################################################################
###############################################################

data_1['KWH_AGUA_SANITARIA'] = (data['Ducha Elec'] + data['Tina Elec'] + data['Lavado loza Elec'] ) /  (data_1['NWEIGHT'])
data_1['KWH_AIRE_ACONDICIONADO'] = (data['A/C Electricidad'] ) /  ( data_1['NWEIGHT'])
data_1['KWH_CALEFACCION'] = (data['Calefacción Central Elec'] +
                                   data['Calefactores Elec']  ) /  ( data_1['NWEIGHT'])

data_1['KWH_COCINA'] = (data['Hornillo elec. Electricidad'] ) /  ( data_1['NWEIGHT'])

data_1['KWH_ILUMINACION'] = (data['Iluminación Electricidad'] ) /  ( data_1['NWEIGHT'])

data_1['KWH_LAVADO'] = (data['Lavado Ropa Elec']+data['Secado Ropa Elec'] ) /  (data_1['NWEIGHT'])
data_1['KWH_REFRIGERACION'] = (data['Refrigerador Electricidad']+data['Freezer Electricidad'] ) /  ( data_1['NWEIGHT'])
data_1['KWH_TV'] = (data['TV Electricidad'] ) /  ( data_1['NWEIGHT'])

vars = [   'Hervidor Electricidad','Microondas Electricidad',
       'Plancha Electricidad','Aspiradora Electricidad','Computador Electricidad',
       'Juegos Electricidad','Stand by Electricidad',
       'Cafetera Electricidad','Bomba de riego Electricidad','Piscina Electricidad','Otros Electricidad']
data_1['KWH_OTROS'] = (data[vars].sum(axis=1) ) /  ( data_1['NWEIGHT'])


## CARGA EL MODELO

In [11]:
MODELO_CHILE = {
     'const': -35.28135312449055,
    'CALEFACCION_electrico': -0.7849438634089401,
    'AIRE_ACONDICIONADO': 0.5332813348260148,
    'AGUA_CALIENTE_electrica': 0.02233944840823343,
    'VENTANAS_log': 0.8915352020395764,
    'DORMITORIOS_log': 1.1286927099564847,
    'HABITANTES_totales_log': 0.6943882161221977,
    'CLIMA_grados_dias_calefaccion_log': 23.370819228842834,
    'CLIMA_grados_dias_enfriamiento_log': -0.4461417276925326,
    'AIRE_ACONDICIONADO_X_CLIMA_grados_dias_enfriamiento_log': -0.20317563962628898,
    'CALEFACCION_electrico_X_CLIMA_grados_dias_calefaccion_log': 0.27759756806618174,
    'CLIMA_grados_dias_calefaccion_log_2': -3.6241690026815316,
    'CLIMA_grados_dias_enfriamiento_log_2': 0.2113218394637681,
    'VENTANAS_log_X_DORMITORIOS_log': -0.45814324890253655,
    'VENTANAS_log_X_CLIMA_grados_dias_enfriamiento_log': -0.24739716669808576,
    'DORMITORIOS_log_X_HABITANTES_totales_log': -0.8606681093577948}

## APLICA EL MODELO

In [12]:
data_1['const'] = 0
prediccion = data_1['const']
data_1['const'] = 1
for var in MODELO_CHILE.keys():
  prediccion += MODELO_CHILE[var] * data_1[var]

data_1['PREDICCION'] = 10**prediccion

In [13]:
data_1['Eficiencia'] = 'Muy alta'
data_1['Consumo'] = 'Muy bajo'

data_1.loc[data_1["PREDICCION"] >  1018, 'Consumo']= 'Bajo'
data_1.loc[data_1["PREDICCION"] >  1465, 'Consumo']= 'Moderado'
data_1.loc[data_1["PREDICCION"] >  1918, 'Consumo']= 'Alto'
data_1.loc[data_1["PREDICCION"] >  2706, 'Consumo']= 'Muy alto'


z = ( np.log10(1+data_1["KWH"]) - np.log10(1+data_1["PREDICCION"]) ) / 0.2504
data_1.loc[z > -0.84, 'Eficiencia']= 'Alta'
data_1.loc[z > -0.25, 'Eficiencia']= 'Moderada'
data_1.loc[z > 0.25, 'Eficiencia']= 'Baja'
data_1.loc[z > 0.84, 'Eficiencia']= 'Muy baja'




data_1['AHORRO']  = 0.0


# ============================================================
# RECOMENDACIONES ESPECÍFICAS
# ============================================================
# ------------------------------------------------------------
# AGUA CALIENTE
# ------------------------------------------------------------
aux =  data_1['AGUA_CALIENTE_electrica'] == 1
data_1.loc[aux, 'AHORRO'] += data_1.loc[aux, 'KWH_AGUA_SANITARIA'] * 0.20

# ------------------------------------------------------------
# AIRE ACONDICIONADO
# ------------------------------------------------------------
aux =  data_1['KWH_AIRE_ACONDICIONADO'] > 0.1 * data_1['KWH']
data_1.loc[aux, 'AHORRO'] += data_1.loc[aux, 'KWH_AIRE_ACONDICIONADO'] * 0.15

# ------------------------------------------------------------
# CALEFACCIÓN
# ------------------------------------------------------------
aux =  data_1['KWH_CALEFACCION'] > 0.1 * data_1['KWH']
data_1.loc[aux, 'AHORRO'] += data_1.loc[aux, 'KWH_CALEFACCION'] * 0.20

# ------------------------------------------------------------
# VENTANAS
# ------------------------------------------------------------
aux = (10**data_1['VENTANAS_log'] - 1) > (3 * (10**data_1['DORMITORIOS_log'] - 1))
data_1.loc[aux, 'AHORRO'] += data_1.loc[aux, 'KWH_CALEFACCION'] * 0.25 + data_1.loc[aux, 'KWH_AIRE_ACONDICIONADO'] * 0.15

# ------------------------------------------------------------
# SECARROPAS / LAVADO
# ------------------------------------------------------------

aux = (data_1['SECARROPAS_electrico'] == 1) & (data_1['LAVARROPAS_frecuencia_log'] >= np.log10(2.5+1))
data_1.loc[aux, 'AHORRO'] += data_1.loc[aux, 'KWH_LAVADO'] * 0.5

# ------------------------------------------------------------
# STANDBY — SOLO CUANDO NO HAY GRANDES CONSUMIDORES
# ------------------------------------------------------------
aux = (data_1['AIRE_ACONDICIONADO']==0) & (data_1['CALEFACCION_electrico']==0) & (data_1['AGUA_CALIENTE_electrica']==0)
data_1.loc[aux, 'AHORRO'] += data_1.loc[aux, 'KWH']*0.10

#--------------------------------------------------
# ILUMINACIÓN
# ------------------------------------------------------------
aux = data_1['KWH_ILUMINACION'] > 0.1 * data_1['KWH']
data_1.loc[aux, 'AHORRO'] += data_1.loc[aux, 'KWH_ILUMINACION'] * 0.5

#-------------------------------------------------
# REFRIGERACION
#-------------------------------------------------

aux = data_1['KWH_REFRIGERACION'] > 0.1 * data_1['KWH']
data_1.loc[aux, 'AHORRO'] += data_1.loc[aux, 'KWH_REFRIGERACION'] * 0.3

ref = 10**data_1['REFRIGERADOR_log'] - 1
frz = 10**data_1['FREEZER_log'] - 1
aux = (data_1['KWH_REFRIGERACION'] > 0.1 * data_1['KWH']) & ((ref>1.5) |  (frz>0.5) )
data_1.loc[aux, 'AHORRO'] += ( ref.loc[aux] + frz.loc[aux] - 1 ) * 400












## AHORRO POR GRUPO

In [21]:
def get_ahorro(data, consumo, eficiencia, fun='sum'):
  if isinstance(consumo, str):
    consumo = [consumo]
  if isinstance(eficiencia, str):
    eficiencia = [eficiencia]
  temp_data = data.loc[(data['Consumo'].isin(consumo) ) & (data['Eficiencia'].isin(eficiencia)), ['AHORRO', 'NWEIGHT']]
  out = temp_data['AHORRO'] * temp_data['NWEIGHT']
  if fun == 'sum':
    out = out.sum()
  elif fun == 'mean':
    out = out.sum() / temp_data['NWEIGHT'].sum()
  else:
    print('Función no reconocida')

  return out

print("AHORRO POR GRUPO")
print("--------------")
print("CONSUMO BAJO")
print( f"Ineficientes: US${ get_ahorro(data_1, ["Muy bajo", "Bajo"], ["Muy baja", "Baja"], 'mean')*0.228:,.0f}")
print("CONSUMO MEDIO")
print( f"Ineficientes: US${ get_ahorro(data_1, ["Moderado"], ["Muy baja", "Baja"], 'mean')*0.228:,.0f}")
print("CONSUMO ALTO")
print( f"Ineficientes: US${ get_ahorro(data_1, ["Muy alto", "Alto"], ["Muy baja", "Baja"], 'mean')*0.228:,.0f}")


AHORRO POR GRUPO
--------------
CONSUMO BAJO
Ineficientes: 150
CONSUMO MEDIO
Ineficientes: 229
CONSUMO ALTO
Ineficientes: 298


## AHORRO PAIS

In [20]:

print("AHORRO PAIS")
print("--------------")
print("TOTAL")
print( f"Ineficientes: US${ get_ahorro(data_1, ["Muy bajo", "Bajo", "Moderado","Muy alto", "Alto"], ["Muy baja", "Baja"], 'sum')*0.228:,.0f}")
print("--------------")
print("CONSUMO BAJO")
print( f"Ineficientes: US${ get_ahorro(data_1, ["Muy bajo", "Bajo"], ["Muy baja", "Baja"], 'sum')*0.228:,.0f}")
print("-------------")
print("CONSUMO MEDIO")
print( f"Ineficientes: US${ get_ahorro(data_1, ["Moderado"], ["Muy baja", "Baja"], 'sum')*0.228:,.0f}")
print("-------------")
print("CONSUMO ALTO")
print( f"Ineficientes: US${ get_ahorro(data_1, ["Alto", "Muy alto"], ["Muy baja", "Baja"], 'sum')*0.228:,.0f}")


AHORRO PAIS
--------------
TOTAL
Ineficientes: 561,407,698
--------------
CONSUMO BAJO
Ineficientes: 87,411,026
-------------
CONSUMO MEDIO
Ineficientes: 232,971,931
-------------
CONSUMO ALTO
Ineficientes: 241,024,742
